In [1]:
!pip install -q arxiv pymupdf 

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 55.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 93.7 MB/s eta 0:00:00:00:01


In [2]:
import arxiv
import fitz
import json
import os
import time

In [3]:
client = arxiv.Client()

queries = [
    "large language models transformer",
    "retrieval augmented generation RAG",
    "fine-tuning attention mechanisms"
]

all_results = []
seen_ids = set()

for query in queries:
    search = arxiv.Search(query=query, max_results=170)  # was 34 → ~510 raw, ~450-500 after dedup
    results = list(client.results(search))
    for r in results:
        if r.entry_id not in seen_ids:
            seen_ids.add(r.entry_id)
            all_results.append(r)
    time.sleep(3)

print(f"Total unique papers found: {len(all_results)}")


Total unique papers found: 505


In [4]:
import requests

os.makedirs('/kaggle/working/papers', exist_ok=True)

downloaded = []
failed = []

headers = {'User-Agent': 'Mozilla/5.0 (compatible; ResearchLens/1.0)'}

for paper in all_results:
    paper_id = paper.entry_id.split('/')[-1]
    pdf_path = f'/kaggle/working/papers/{paper_id}.pdf'
    
    if os.path.exists(pdf_path):
        downloaded.append({'id': paper_id, 'path': pdf_path, 'title': paper.title})
        continue
    
    try:
        url = f'https://arxiv.org/pdf/{paper_id}'
        response = requests.get(url, headers=headers, timeout=30)
        
        if response.status_code == 200 and len(response.content) > 10000:
            with open(pdf_path, 'wb') as f:
                f.write(response.content)
            downloaded.append({'id': paper_id, 'path': pdf_path, 'title': paper.title})
            print(f"✓ {paper_id}")
        else:
            failed.append({'id': paper_id, 'error': f'status {response.status_code}'})
            print(f"✗ {paper_id} — status {response.status_code}")
        
        time.sleep(2)
        
    except Exception as e:
        failed.append({'id': paper_id, 'error': str(e)})
        print(f"✗ {paper_id} — {str(e)}")

print(f"\nDownloaded: {len(downloaded)}")
print(f"Failed: {len(failed)}")

✓ 2501.05032v2
✓ 2402.14679v2
✓ 2405.11357v3
✓ 2403.09676v1
✓ 2107.03844v3
✓ 2309.02144v1
✓ 2407.01505v1
✓ 2310.00905v2
✓ 2407.08029v1
✓ 2312.05434v1
✓ 2512.06483v1
✓ 2306.13549v4
✓ 2402.14800v2
✓ 2404.16891v1
✓ 2311.04589v3
✓ 2110.08975v2
✓ 2305.03715v1
✓ 2402.05964v2
✓ 2412.16117v1
✓ 2509.16679v1
✓ 2302.05406v1
✓ 2305.14982v2
✓ 2403.09832v1
✓ 2601.08844v1
✓ 2606.21645v1
✓ 2312.12141v4
✓ 2604.14214v1
✓ 2503.21676v2
✓ 2409.15256v1
✓ 2507.12472v1
✓ 2312.12006v1
✓ 2603.22158v1
✓ 2604.04450v1
✓ 2402.12065v2
✓ 2508.04848v1
✓ 2411.16260v3
✓ 2309.01446v4
✓ 2310.14025v1
✓ 2406.16167v1
✓ 2412.11937v1
✓ 2308.16118v2
✓ 2310.05189v2
✓ 2510.21459v1
✓ 2512.12447v1
✓ 2408.16440v2
✓ 2402.11651v2
✓ 2502.04355v1
✓ 2606.19264v1
✓ 2512.08480v1
✓ 2308.05502v2
✓ 2312.10793v3
✓ 2405.13828v2
✓ 2409.02228v1
✓ 2306.15498v1
✓ 2504.19565v3
✓ 2603.21389v1
✓ 2503.16581v1
✓ 2402.04477v1
✓ 2304.12244v3
✓ 2409.11353v3
✓ 2410.12096v1
✓ 2408.04646v2
✓ 2502.05945v3
✓ 2510.21894v1
✓ 2603.01691v1
✓ 2502.09173v1
✓ 2309.137

In [5]:
def parse_pdf(pdf_path, paper_id, title):
    doc = fitz.open(pdf_path)
    pages = []
    
    for page_num, page in enumerate(doc):
        text = page.get_text()
        if len(text.strip()) > 50:
            pages.append({
                'paper_id': paper_id,
                'title': title,
                'page_num': page_num,
                'text': text.strip()
            })
    
    doc.close()
    return pages

all_pages = []
parse_errors = []

for paper in downloaded:
    try:
        pages = parse_pdf(paper['path'], paper['id'], paper['title'])
        all_pages.extend(pages)
    except Exception as e:
        parse_errors.append({'id': paper['id'], 'error': str(e)})

print(f"Total pages parsed: {len(all_pages)}")
print(f"Parse errors: {len(parse_errors)}")
print(f"Sample page text (first 300 chars):")
print(all_pages[0]['text'][:300])

MuPDF error: syntax error: could not parse color space (602 0 R)

MuPDF error: syntax error: cannot find ExtGState resource 'pgf@ca1.0'

MuPDF error: syntax error: cannot find ExtGState resource 'pgf@ca1.0'

MuPDF error: syntax error: cannot find ExtGState resource 'pgf@ca1.0'

MuPDF error: syntax error: cannot find ExtGState resource 'pgf@ca1.0'

MuPDF error: syntax error: cannot find ExtGState resource 'pgf@ca1.0'

MuPDF error: syntax error: cannot find ExtGState resource 'pgf@ca1.0'

MuPDF error: syntax error: cannot find ExtGState resource 'pgf@ca1.0'

MuPDF error: syntax error: cannot find ExtGState resource 'pgf@ca1.0'

MuPDF error: syntax error: cannot find ExtGState resource 'pgf@ca1.0'

MuPDF error: syntax error: cannot find ExtGState resource 'pgf@ca1.0'

MuPDF error: syntax error: cannot find ExtGState resource 'pgf@ca1.0'

MuPDF error: syntax error: cannot find ExtGState resource 'pgf@ca1.0'

MuPDF error: syntax error: cannot find ExtGState resource 'pgf@ca1.0'

MuPDF error

In [6]:
with open('/kaggle/working/parsed_papers.json', 'w') as f:
    json.dump(all_pages, f)

print(f"Saved {len(all_pages)} pages to parsed_papers.json")

Saved 8565 pages to parsed_papers.json


In [7]:
metadata = []
for paper in all_results:
    paper_id = paper.entry_id.split('/')[-1]
    metadata.append({
        'id': paper_id,
        'title': paper.title,
        'authors': [a.name for a in paper.authors],
        'abstract': paper.summary,
        'published': str(paper.published),
        'url': paper.entry_id
    })

with open('/kaggle/working/papers_metadata.json', 'w') as f:
    json.dump(metadata, f)

print(f"Saved metadata for {len(metadata)} papers")

Saved metadata for 505 papers


In [8]:
!pip install -q "datasets==2.20.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 547.8/547.8 kB 11.0 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.1/316.1 kB 14.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
tpot 1.1.0 requires dill>=0.3.9, but you have dill 0.3.8 which is incompatible.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.5.0 which is incompatible.


In [9]:
!pip show pyarrow | grep Version

Version: 24.0.0


In [10]:
from datasets import load_dataset

In [11]:
qasper = load_dataset("allenai/qasper")

print(f"QASPER splits: {qasper.keys()}")
print(f"Train size: {len(qasper['train'])}")
print(f"Validation size: {len(qasper['validation'])}")
print(f"Test size: {len(qasper['test'])}")
print(f"\nSample entry keys: {qasper['train'][0].keys()}")

Generating train split:   0%|          | 0/888 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/281 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/416 [00:00<?, ? examples/s]

QASPER splits: dict_keys(['train', 'validation', 'test'])
Train size: 888
Validation size: 281
Test size: 416

Sample entry keys: dict_keys(['id', 'title', 'abstract', 'full_text', 'qas', 'figures_and_tables'])


In [12]:
sample = qasper['train'][0]
print(f"Top level keys: {sample.keys()}")
print(f"\nqas type: {type(sample['qas'])}")
print(f"\nqas sample: {sample['qas']}")

Top level keys: dict_keys(['id', 'title', 'abstract', 'full_text', 'qas', 'figures_and_tables'])

qas type: <class 'dict'>

qas sample: {'question': ['What is the seed lexicon?', 'What are the results?', 'How are relations used to propagate polarity?', 'How big is the Japanese data?', 'What are labels available in dataset for supervision?', 'How big are improvements of supervszed learning results trained on smalled labeled data enhanced with proposed approach copared to basic approach?', 'How does their model learn using mostly raw data?', 'How big is seed lexicon used for training?', 'How large is raw corpus used for training?'], 'question_id': ['753990d0b621d390ed58f20c4d9e4f065f0dc672', '9d578ddccc27dd849244d632dd0f6bf27348ad81', '02e4bf719b1a504e385c35c6186742e720bcb281', '44c4bd6decc86f1091b5fc0728873d9324cdde4e', '86abeff85f3db79cf87a8c993e5e5aa61226dc98', 'c029deb7f99756d2669abad0a349d917428e9c12', '39f8db10d949c6b477fa4b51e7c184016505884f', 'd0bc782961567dc1dd7e074b621a6d6be44b

In [13]:
def extract_qa_pairs(dataset_split, max_pairs=200):
    pairs = []
    
    for entry in dataset_split:
        title = entry['title']
        questions = entry['qas']['question']
        answers_list = entry['qas']['answers']
        
        for question, answer_obj in zip(questions, answers_list):
            for answer in answer_obj['answer']:
                if answer['unanswerable']:
                    continue
                
                free_form = answer['free_form_answer']
                if free_form and len(free_form) > 10:
                    pairs.append({
                        'question': question,
                        'answer': free_form,
                        'paper_title': title
                    })
                    break
            
            if len(pairs) >= max_pairs:
                return pairs
    
    return pairs

qasper_pairs = extract_qa_pairs(qasper['train'], max_pairs=200)

print(f"Extracted {len(qasper_pairs)} QA pairs from QASPER")
print(f"\nSample pair:")
print(f"Q: {qasper_pairs[0]['question']}")
print(f"A: {qasper_pairs[0]['answer'][:200]}")

Extracted 200 QA pairs from QASPER

Sample pair:
Q: What is the seed lexicon?
A: a vocabulary of positive and negative predicates that helps determine the polarity score of an event


In [14]:
with open('/kaggle/working/qasper_pairs.json', 'w') as f:
    json.dump(qasper_pairs, f)

print(f"Saved {len(qasper_pairs)} QASPER pairs to qasper_pairs.json")

Saved 200 QASPER pairs to qasper_pairs.json


In [15]:
files_to_check = [
    '/kaggle/working/parsed_papers.json',
    '/kaggle/working/papers_metadata.json',
    '/kaggle/working/qasper_pairs.json',
]

for filepath in files_to_check:
    if os.path.exists(filepath):
        size_mb = os.path.getsize(filepath) / (1024 * 1024)
        print(f"✓ {filepath.split('/')[-1]} — {size_mb:.2f} MB")
    else:
        print(f"✗ MISSING: {filepath}")

print(f"\nTotal papers directory:")
pdf_count = len([f for f in os.listdir('/kaggle/working/papers') if f.endswith('.pdf')])
print(f"✓ {pdf_count} PDFs in /kaggle/working/papers/")

✓ parsed_papers.json — 33.57 MB
✓ papers_metadata.json — 0.77 MB
✓ qasper_pairs.json — 0.06 MB

Total papers directory:
✓ 497 PDFs in /kaggle/working/papers/
